In [1]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [2]:
df = pd.read_csv('DaftarSaham.csv')


In [3]:
df.columns


Index(['Code', 'Name', 'ListingDate', 'Shares', 'ListingBoard', 'Sector',
       'LastPrice', 'MarketCap', 'MinutesFirstAdded', 'MinutesLastUpdated',
       'HourlyFirstAdded', 'HourlyLastUpdated', 'DailyFirstAdded',
       'DailyLastUpdated'],
      dtype='object')

In [4]:
df['Sector'].unique()


array(['Consumer Non-Cyclicals', 'Consumer Cyclicals', 'Financials',
       'Industrials', 'Infrastructures', 'Properties & Real Estate',
       'Basic Materials', 'Energy', 'Transportation & Logistic',
       'Technology', 'Healthcare'], dtype=object)

In [5]:
df_financial = df[df['Sector'] == 'Financials']
df_financial


,Code,Name,ListingDate,Shares,ListingBoard,Sector,LastPrice,MarketCap,MinutesFirstAdded,MinutesLastUpdated,HourlyFirstAdded,HourlyLastUpdated,DailyFirstAdded,DailyLastUpdated
2,ABDA,Asuransi Bina Dana Arta Tbk.,1989-07-06,6.208067e+08,Pengembangan,Financials,6700.0,4.159405e+12,2021-11-01 09:00:00,2022-11-11 15:59:00,2020-04-16 09:00:00,2022-11-11 16:00:00,2001-04-16,2023-01-06
9,ADMF,Adira Dinamika Multi Finance T,2004-03-31,1.000000e+09,Utama,Financials,8850.0,8.850000e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2004-03-31,2023-01-06
15,AGRO,Bank Raya Indonesia Tbk.,2003-08-08,2.252005e+10,Utama,Financials,406.0,9.143142e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2003-08-08,2023-01-06
16,AGRS,Bank IBK Indonesia Tbk.,2014-12-22,1.748160e+10,Pengembangan,Financials,89.0,1.555863e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2014-12-22,2023-01-06
17,AHAP,Asuransi Harta Aman Pratama Tb,1990-09-14,2.940000e+09,Pengembangan,Financials,64.0,1.881600e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2001-04-16,2023-01-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,VINS,Victoria Insurance Tbk.,2015-09-28,1.460574e+09,Pengembangan,Financials,133.0,1.942563e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2015-09-28,2023-01-06
800,VRNA,Verena Multi Finance Tbk.,2008-06-25,5.687354e+09,Pengembangan,Financials,99.0,5.630480e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2008-06-25,2023-01-06
801,VTNY,Venteny Fortuna International,2022-12-15 00:00:00,6.265193e+09,Pengembangan,Financials,444.0,2.781746e+12,2022-12-15 09:00:00,2023-01-06 15:59:00,2022-12-15 09:00:00,2023-01-06 15:00:00,2022-12-15,2023-01-06
815,WOMF,Wahana Ottomitra Multiartha Tb,2004-12-13,3.481481e+09,Utama,Financials,256.0,8.912593e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2004-12-13,2023-01-06


In [6]:
df_financial.shape

(106, 14)

In [7]:
import os

folder_path = "daily"

all_files = os.listdir(folder_path)


In [8]:
financial_codes = df_financial['Code'].tolist()

In [9]:
financial_files = [
    f for f in all_files 
    if f.replace(".csv", "") in financial_codes
]


In [10]:
data_financial = []

for file in financial_files:
    path = os.path.join(folder_path, file)
    df_temp = pd.read_csv(path)
    df_temp['Code'] = file.replace(".csv", "")
    data_financial.append(df_temp)

data_financial = pd.concat(data_financial, ignore_index=True)


In [11]:
data_financial


,timestamp,open,low,high,close,volume,Code
0,2001-04-16,194,152,194,152,0,ABDA
1,2001-04-17,194,152,194,152,0,ABDA
2,2001-04-18,194,152,194,152,0,ABDA
3,2001-04-19,194,152,194,152,0,ABDA
4,2001-04-20,194,152,194,152,0,ABDA
...,...,...,...,...,...,...,...
402615,2023-01-02,2230,2230,2230,2230,0,YULE
402616,2023-01-03,2230,2230,2230,2230,0,YULE
402617,2023-01-04,2230,2230,2230,2230,500,YULE
402618,2023-01-05,2230,2230,2230,2230,0,YULE


In [12]:
H=14

In [13]:
data_financial["future_close"] = data_financial["close"].shift(-H)
data_financial["future_return"] = (data_financial["future_close"] - data_financial["close"]) / data_financial["close"]


In [14]:
buy_threshold = 0.03   # +3% over 14 days
sell_threshold = -0.03 # -3% over 14 days

data_financial["label"] = 0 # hold

data_financial.loc[data_financial["future_return"] > buy_threshold, "label"] = 1 # buy
data_financial.loc[data_financial["future_return"] < sell_threshold, "label"] = -1 # sell

In [15]:
data_financial = data_financial.dropna()

In [16]:

data_financial
data_financial["timestamp"] = pd.to_datetime(data_financial["timestamp"])

C:\Users\dr. Swany J.T\AppData\Local\Temp\ipykernel_25388\2698409068.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_financial["timestamp"] = pd.to_datetime(data_financial["timestamp"])


In [17]:
target_date = "2024/12/31"
target_date =  pd.to_datetime(target_date, format='%Y/%m/%d')

last_date = data_financial["timestamp"].max()
steps = (target_date - last_date).days
print(steps)




725


In [18]:
def level_return(future_return, buy_threshold=0.03, sell_threshold=-0.03):
    if future_return > buy_threshold:
        return "High"
    elif future_return < sell_threshold:
        return "Low"
    else:
        return "Medium"
   

In [26]:
close = data_financial["close"]

min_price = close.min()
max_price = close.max()

D1 =  min_price - 0.05 * min_price
D2 = min_price + 0.05 * min_price

close

0          152
1          152
2          152
3          152
4          152
          ... 
402601    2100
402602    2100
402603    2100
402604    2100
402605    2100
Name: close, Length: 402606, dtype: int64

In [28]:
n_interval =  30 
interval_length = (D2 - D1) / n_interval
intervals = []
for i in range(n_interval):
    lower= D1 + i * interval_length
    upper = lower + interval_length
    intervals.append((lower, upper))
intervals

[(np.float64(4.75), np.float64(4.766666666666667)),
 (np.float64(4.766666666666667), np.float64(4.783333333333333)),
 (np.float64(4.783333333333333), np.float64(4.8)),
 (np.float64(4.8), np.float64(4.816666666666666)),
 (np.float64(4.816666666666666), np.float64(4.833333333333333)),
 (np.float64(4.833333333333333), np.float64(4.85)),
 (np.float64(4.85), np.float64(4.866666666666666)),
 (np.float64(4.866666666666666), np.float64(4.883333333333333)),
 (np.float64(4.883333333333334), np.float64(4.9)),
 (np.float64(4.9), np.float64(4.916666666666667)),
 (np.float64(4.916666666666667), np.float64(4.933333333333334)),
 (np.float64(4.933333333333334), np.float64(4.95)),
 (np.float64(4.95), np.float64(4.966666666666667)),
 (np.float64(4.966666666666667), np.float64(4.983333333333333)),
 (np.float64(4.983333333333333), np.float64(5.0)),
 (np.float64(5.0), np.float64(5.016666666666667)),
 (np.float64(5.016666666666667), np.float64(5.033333333333333)),
 (np.float64(5.033333333333333), np.float64(

In [27]:
def fuzzify(value):
    if np.isnan(value):
        return None
    for i,(low,high) in enumerate(intervals):
        if low <= value < high:
            return i 
    return None
fuzzy_series = [fuzzify(x) for x in close]    
fuzzy_series 


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [22]:
FLRG ={}

for i in range(len(fuzzy_series)-1):
    current_state = fuzzy_series[i]
    next_state = fuzzy_series[i+1]
    if current_state not in FLRG:
        FLRG[current_state] = []
    FLRG[current_state].append(next_state)

In [23]:
last_state = fuzzy_series[-1]

if last_state is not None and last_state in FLRG:
    next_states = [s for s in FLRG[last_state] if s is not None]
    if next_states:
        predicted_state = int(np.mean(next_states))
    else:
        predicted_state = last_state
else:
    predicted_state = last_state

print(predicted_state)

None
